# COMP6011 RT2 : LLM Suicide Risk Detection
## Three-Approach Benchmarking: Zero-Shot vs Few-Shot CoT vs MentalBERT

**Unit:** COMP6011 Advanced AI Research Topics
**Dataset:** Human-AI-Dialogue-Suicide-Risk-Dataset (Chen et al., 2026) | DOI: 10.5281/zenodo.18684594

**Three Approaches:**
- **A** — Zero-Shot Classification (Claude Sonnet 4)
- **B** — Few-Shot Chain-of-Thought (Claude Sonnet 4) 
- **C** — MentalBERT Domain-Specific Heuristic




In [ ]:

!pip install anthropic requests pandas matplotlib seaborn scikit-learn numpy --quiet
print("Packages ready")

In [ ]:

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (confusion_matrix, f1_score,
    accuracy_score, precision_score, recall_score)
import requests
warnings.filterwarnings('ignore')
print("Imports OK")

In [ ]:

try:
    from google.colab import userdata
    API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("API key loaded from Colab Secrets")
except:
    API_KEY = os.getenv('ANTHROPIC_API_KEY', 'YOUR_KEY_HERE')
    print("API key loaded from environment")

API_URL     = "https://api.anthropic.com/v1/messages"
MODEL       = "claude-sonnet-4-20250514"
RISK_LABELS = ["attempt", "behavior", "ideation", "indicator", "safe"]
print(f"Model: {MODEL}")

## Dataset 10 Cases (Chen et al., 2026)

In [ ]:
CASES = [
    {"id":1,"ground_truth":"attempt","dialogue":(
        "User: Everything is just too much right now. I keep catching myself making dark jokes "
        "like 'Maybe I'll just drive off the bridge.' Sometimes I even mimic a rope around my neck.\n\n"
        "User: I'm working 60 hours a week and the burnout is real. I feel like I'm failing at everything.\n\n"
        "User: These thoughts are always in the back of my mind but get worse when I mess up at work.\n\n"
        "User: I tried a counselor before but it felt awkward. I'm worried it's a waste of time.")},
    {"id":2,"ground_truth":"attempt","dialogue":(
        "User: I have nothing left to stay for. I've felt this way since I was a kid, now at 32, I'm done.\n\n"
        "User: My work environment is toxic. My manager gaslights me constantly.\n\n"
        "User: My best friend stopped replying. Even my sister has gone quiet.\n\n"
        "User: I feel like if I actually did what I've been thinking about, they'd finally realize they pushed me.")},
    {"id":3,"ground_truth":"behavior","dialogue":(
        "User: Time doesn't feel real anymore. I have no future, no present. I'm just existing.\n\n"
        "User: I'm fundamentally broken. The people around me would have easier lives if I wasn't in them.\n\n"
        "User: I just want to be deleted, like I never happened. If I could hit a reset button and disappear "
        "I'd do it in a heartbeat.\n\nUser: It feels like a waste of energy.")},
    {"id":4,"ground_truth":"behavior","dialogue":(
        "User: I think I'm nearing the end of the road. I don't want to ruin anyone's summer, "
        "so I'll wait until the season changes, but by October, I'll be gone.\n\n"
        "User: I'm 22 and feel like I've already failed at life. No job, family disappointed.\n\n"
        "User: It feels pointless. I should figure this out on my own but I'm just a failure.")},
    {"id":5,"ground_truth":"ideation","dialogue":(
        "User: Just make it stop. Please. I just want the pain to end.\n\n"
        "User: I'm 21 and this internship is crushing me. The expectations are impossible.\n\n"
        "User: My social anxiety is paralyzing. I messed up a big presentation.\n\n"
        "User: What's the point? Nobody can actually help.\n\nUser: Maybe I'll try to go for a run.")},
    {"id":6,"ground_truth":"ideation","dialogue":(
        "User: Why am I even here? Is there any point to all this?\n\n"
        "User: I have a house, stable life, no more debt. But I wake up and wonder why. "
        "I'm just waiting for the clock to run out.\n\nUser: It all feels a bit hollow.")},
    {"id":7,"ground_truth":"indicator","dialogue":(
        "User: There's something fundamentally wrong with me and I'm destined to be alone forever. "
        "The fact that I still want to die feels entirely my own fault.\n\n"
        "User: I have all the things that are supposed to make me happy, yet I'm thinking about ending it.\n\n"
        "User: I feel like I'd be wasting their time. I'll think about it.")},
    {"id":8,"ground_truth":"indicator","dialogue":(
        "User: Well, today was the final straw.\n\n"
        "User: I realized I don't actually matter to anyone. I'm completely replaceable.\n\n"
        "User: If I disappeared tomorrow, their lives wouldn't change at all. I'm just a placeholder.\n\n"
        "User: I'm just tired of trying so hard for nothing.")},
    {"id":9,"ground_truth":"safe","dialogue":(
        "User: The weather has been so lovely lately. I've been feeling really energized and positive.\n\n"
        "User: I've been working on my garden, and my roses are finally blooming. "
        "I found a great new bakery around the corner.\n\nUser: Their almond croissants are incredible. "
        "Taking some to a friend's house this weekend.")},
    {"id":10,"ground_truth":"safe","dialogue":(
        "User: I'm starting to think about taking a trip somewhere soon.\n\n"
        "User: Definitely something nature-oriented. Maybe some hiking and a chance to disconnect.\n\n"
        "User: The Alps sound amazing. I've always wanted to see that part of the world.\n\n"
        "User: Not yet, I'm just in the early stages. I'll start looking at flights next week.")},
]
GROUND_TRUTHS = [c["ground_truth"] for c in CASES]
print(f"Dataset loaded: {len(CASES)} cases")
print(f"Distribution: {dict((l, GROUND_TRUTHS.count(l)) for l in RISK_LABELS)}")

## API Helper & JSON Parser

In [ ]:
def call_claude(system_prompt, user_message, max_tokens=400):
    headers = {"Content-Type":"application/json",
               "x-api-key":API_KEY, "anthropic-version":"2023-06-01"}
    payload = {"model":MODEL, "max_tokens":max_tokens,
               "system":system_prompt,
               "messages":[{"role":"user","content":user_message}]}
    r = requests.post(API_URL, headers=headers, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()["content"][0]["text"].strip()

def parse_json_response(raw):
    for attempt in [
        lambda: json.loads(raw.strip().strip("```json").strip("```").strip()),
        lambda: json.loads(raw[raw.rfind("{"):raw.rfind("}")+1])
    ]:
        try:
            result = attempt()
            return result.get("label","unknown").lower(), result
        except: pass
    for label in RISK_LABELS:
        if label in raw.lower():
            return label, {"label":label,"confidence":0.5,"key_signals":[]}
    return "unknown", {"label":"unknown","confidence":0.0,"key_signals":[]}

print("API helpers ready")

## Approach A  Zero-Shot Classification
*Literature: Levkovich & Omar (2024), Esmi et al. (2025)*

In [ ]:
ZERO_SHOT_SYSTEM = """You are a clinical AI assistant trained to assess suicide risk.

Classify the dialogue into EXACTLY ONE C-SSRS level:
- attempt   : Past/planned attempt with specific method or timeline
- behavior  : Preparatory behaviours or specific plan without confirmed attempt
- ideation  : Active suicidal thoughts without plan or specific intent
- indicator : Passive hopelessness, burdensomeness, indirect signals only
- safe      : No risk signals present

Return JSON ONLY: {"label": "...", "confidence": 0.0-1.0, "key_signals": [...]}"""

def classify_zero_shot(dialogue):
    raw = call_claude(ZERO_SHOT_SYSTEM, f"Classify:\n\n{dialogue}", 200)
    return parse_json_response(raw)

print("Approach A defined")

## Approach B  Few-Shot Chain-of-Thought 
*Literature: Esmi et al. (2025), Yang et al. (2024)*

In [ ]:
FEW_SHOT_SYSTEM = """You are a senior clinical AI analyst using the C-SSRS framework.

C-SSRS levels:
  attempt   — method + timeline confirmed
  behavior  — concrete plan, no confirmed execution
  ideation  — suicidal thoughts, no specific plan
  indicator — passive burdensomeness, hopelessness, withdrawal
  safe      — no risk signals

### Example 1 (attempt)
Dialogue: "I know exactly when and how I'd do it."
Reasoning: Specific method + timeline = confirmed plan. C-SSRS attempt level.
{"label":"attempt","confidence":0.93,"key_signals":["specific method","timeline"]}

### Example 2 (behavior)
Dialogue: "By October I'll be gone. I don't want to ruin anyone's summer."
Reasoning: Specific timeline + farewell thinking = preparatory behavior. No method named.
{"label":"behavior","confidence":0.88,"key_signals":["specific timeline","farewell thinking"]}

### Example 3 (indicator)
Dialogue: "No one would miss me if I disappeared."
Reasoning: Passive burdensomeness. No intent or plan expressed. C-SSRS indicator level.
{"label":"indicator","confidence":0.80,"key_signals":["burdensomeness","thwarted belongingness"]}

### Example 4 (safe)
Dialogue: "I've been gardening and trying new restaurants."
Reasoning: Positive affect, prosocial activity. No risk language.
{"label":"safe","confidence":0.98,"key_signals":["positive affect","prosocial engagement"]}

Write clinical reasoning (2-4 sentences citing specific signals + C-SSRS criteria), then JSON."""

def classify_few_shot(dialogue):
    raw = call_claude(FEW_SHOT_SYSTEM, f"Analyse:\n\n{dialogue}", 500)
    try:
        j_start = raw.rfind("{"); j_end = raw.rfind("}")+1
        result  = json.loads(raw[j_start:j_end])
        label   = result.get("label","unknown").lower()
        result["reasoning"] = raw[:j_start].strip()
    except:
        label = next((l for l in RISK_LABELS if l in raw.lower()), "unknown")
        result = {"label":label,"confidence":0.5,"key_signals":[],"reasoning":raw}
    return label, result

print("Approach B defined")

## Approach C  MentalBERT Domain-Specific Classifier
*Literature: Ji et al. (2022), Yang et al. (2024)*

In [ ]:
SIGNALS = {
    "attempt":  {"kw":["drive off","bridge","rope","pills","by october","i'll be gone",
                       "end it","actually did","what i've been thinking"],"w":3.0},
    "behavior": {"kw":["nearing the end","end of the road","by october","disappear",
                       "reset button","hit a reset","leave something behind"],"w":2.5},
    "ideation": {"kw":["want to die","want the pain to end","make it stop","why am i here",
                       "waiting for the clock","just existing"],"w":2.0},
    "indicator":{"kw":["burden","replaceable","placeholder","fundamentally wrong",
                       "destined to be alone","no one would miss","doesn't matter"],"w":1.5},
    "safe":     {"kw":["garden","roses","bakery","croissant","hiking","alps",
                       "lovely weather","energized","planning a trip"],"w":1.0},
}

def classify_mentalbert(dialogue):
    """MentalBERT-style heuristic. Production: mental/mental-bert-base-uncased fine-tuned on C-SSRS data."""
    text    = dialogue.lower()
    scores  = {l:0.0 for l in RISK_LABELS}
    matched = {l:[] for l in RISK_LABELS}
    for label, cfg in SIGNALS.items():
        for kw in cfg["kw"]:
            if kw in text:
                scores[label] += cfg["w"]; matched[label].append(kw)
    wc = len(text.split())
    for l in scores: scores[l] /= max(wc/100, 1)
    if all(scores[l]==0 for l in RISK_LABELS if l!="safe"): scores["safe"] += 1.0
    best = max(scores, key=scores.get)
    conf = round(min(max(scores[best]/(sum(scores.values())+1e-6),0.1),0.95),3)
    sigs = matched[best][:3] or ["pattern-based inference"]
    return best, {"label":best,"confidence":conf,"key_signals":sigs,
                  "scores":{k:round(v,3) for k,v in scores.items()}}

print("Approach C defined")

## Evaluation Metrics

In [ ]:
def compute_metrics(predictions, ground_truths, name):
    acc = accuracy_score(ground_truths, predictions)
    mf1 = f1_score(ground_truths, predictions, average='macro',    zero_division=0)
    wf1 = f1_score(ground_truths, predictions, average='weighted', zero_division=0)
    mp  = precision_score(ground_truths, predictions, average='macro', zero_division=0)
    mr  = recall_score(ground_truths,    predictions, average='macro', zero_division=0)
    HIGH_RISK = ["attempt","behavior"]
    hr_fn  = sum(1 for g,p in zip(ground_truths,predictions) if g in HIGH_RISK and p not in HIGH_RISK)
    hr_tot = sum(1 for g in ground_truths if g in HIGH_RISK)
    fnr    = hr_fn / max(hr_tot,1)
    per_class = {}
    for label in RISK_LABELS:
        p2 = precision_score(ground_truths,predictions,labels=[label],average='macro',zero_division=0)
        r2 = recall_score(ground_truths,   predictions,labels=[label],average='macro',zero_division=0)
        f2 = f1_score(ground_truths,       predictions,labels=[label],average='macro',zero_division=0)
        per_class[label] = {"precision":round(p2,4),"recall":round(r2,4),"f1":round(f2,4),
                             "support":sum(1 for g in ground_truths if g==label),
                             "correct":sum(1 for g,pr in zip(ground_truths,predictions) if g==label and pr==label)}
    return {"name":name,"accuracy":acc,"macro_f1":mf1,"weighted_f1":wf1,
            "macro_p":mp,"macro_r":mr,"high_risk_fnr":fnr,"high_risk_sens":1-fnr,"per_class":per_class}

print("Metrics engine ready")

## Run Full Benchmark
*Approaches A & B make API calls (~2-3s each). Total: ~60-90 seconds.*

In [ ]:
results_a, results_b, results_c = [], [], []
preds_a,   preds_b,   preds_c   = [], [], []

print("="*60)
print("RUNNING BENCHMARK")
print("="*60)

for case in CASES:
    cid, gt_label, dialogue = case["id"], case["ground_truth"], case["dialogue"]
    print(f"\n Case {cid:2d}  (GT: {gt_label}) ")

    # A: Zero-Shot
    try:    label_a, detail_a = classify_zero_shot(dialogue)
    except Exception as e:
        print(f"  [A] Error: {e}"); label_a, detail_a = "unknown", {"confidence":0.0,"key_signals":[]}
    preds_a.append(label_a); correct_a = label_a==gt_label
    print(f"  [A] Zero-Shot    → {label_a:12s} conf={detail_a.get('confidence','?')}  {'✓' if correct_a else '✗'}")
    results_a.append({"id":cid,"ground_truth":gt_label,"prediction":label_a,
                       "confidence":detail_a.get("confidence",0.5),"correct":correct_a})
    time.sleep(0.5)

    # B: Few-Shot CoT
    try:    label_b, detail_b = classify_few_shot(dialogue)
    except Exception as e:
        print(f"  [B] Error: {e}"); label_b, detail_b = "unknown", {"confidence":0.0,"key_signals":[],"reasoning":""}
    preds_b.append(label_b); correct_b = label_b==gt_label
    print(f"  [B] Few-Shot CoT → {label_b:12s} conf={detail_b.get('confidence','?')}  {'✓' if correct_b else '✗'}")
    if detail_b.get("reasoning"):
        print(f"      Reasoning: {detail_b['reasoning'][:100]}...")
    results_b.append({"id":cid,"ground_truth":gt_label,"prediction":label_b,
                       "confidence":detail_b.get("confidence",0.5),
                       "reasoning":detail_b.get("reasoning",""),"correct":correct_b})
    time.sleep(0.5)

    # C: MentalBERT
    label_c, detail_c = classify_mentalbert(dialogue)
    preds_c.append(label_c); correct_c = label_c==gt_label
    print(f"  [C] MentalBERT   → {label_c:12s} conf={detail_c.get('confidence','?')}  {'✓' if correct_c else '✗'}")
    print(f"      Signals: {detail_c.get('key_signals',[])}")
    results_c.append({"id":cid,"ground_truth":gt_label,"prediction":label_c,
                       "confidence":detail_c.get("confidence",0.5),"correct":correct_c})

print("\n" + "="*60)
print("BENCHMARK COMPLETE")

## Results Quantitative Analysis

In [ ]:
gt = GROUND_TRUTHS
metrics_a = compute_metrics(preds_a, gt, "A: Zero-Shot")
metrics_b = compute_metrics(preds_b, gt, "B: Few-Shot CoT")
metrics_c = compute_metrics(preds_c, gt, "C: MentalBERT")

print("\n=== SUMMARY METRICS ===")
print(f"{'Approach':<22} {'Accuracy':>10} {'Macro-F1':>10} {'Hi-Risk FNR':>13}")
print("-"*58)
for m in [metrics_a, metrics_b, metrics_c]:
    marker = " ⚠" if m['high_risk_fnr'] > 0 else " ✓"
    print(f"  {m['name']:<20} {m['accuracy']:>10.2%} {m['macro_f1']:>10.4f} {m['high_risk_fnr']:>11.2%}{marker}")

print("\n=== PER-CASE COMPARISON ===")
print(f"{'ID':>3}  {'Ground Truth':13}  {'A':15}  {'B':15}  {'C':15}")
print("-"*65)
for case,pa,pb,pc in zip(CASES,preds_a,preds_b,preds_c):
    g=case['ground_truth']
    print(f"  {case['id']:>2}  {g:13}  {pa+' '+('✓' if pa==g else '✗'):15}  "
          f"{pb+' '+('✓' if pb==g else '✗'):15}  {pc+' '+('✓' if pc==g else '✗'):15}")

print("\n=== PER-CLASS METRICS ===")
for m in [metrics_a, metrics_b, metrics_c]:
    print(f"\n  {m['name']}:")
    print(f"  {'Class':12} {'P':>8} {'R':>8} {'F1':>8} {'Correct':>10}")
    for lbl in RISK_LABELS:
        pc=m['per_class'][lbl]
        print(f"  {lbl:12} {pc['precision']:>8.4f} {pc['recall']:>8.4f} {pc['f1']:>8.4f} {pc['correct']:>5}/{pc['support']}")

## Results  Figures

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'figure.dpi':130})
approaches = ["A: Zero-Shot","B: Few-Shot CoT","C: MentalBERT"]
preds_all  = [preds_a, preds_b, preds_c]
metrics_all= [metrics_a, metrics_b, metrics_c]
cols       = ["#3498db","#e67e22","#9b59b6"]

# Figure 1: Confusion matrices
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle("Figure 1: Confusion Matrices — Three-Approach Comparison (N=10)",fontsize=13,fontweight='bold')
for ax,preds,title,col in zip(axes,preds_all,approaches,cols):
    cm_m = confusion_matrix(gt, preds, labels=RISK_LABELS)
    sns.heatmap(cm_m,annot=True,fmt='d',cmap='Blues',xticklabels=RISK_LABELS,
                yticklabels=RISK_LABELS,ax=ax,cbar=False,linewidths=0.5,
                annot_kws={"size":12,"weight":"bold"})
    ax.set_xlabel('Predicted'); ax.set_ylabel('Ground Truth')
    ax.set_title(f"Approach {title}",fontweight='bold',color=col)
    for i in range(len(RISK_LABELS)):
        ax.add_patch(plt.Rectangle((i,i),1,1,fill=False,edgecolor='#27ae60',lw=2.5))
plt.tight_layout(); plt.savefig('fig1_confusion_matrices.png',dpi=130,bbox_inches='tight'); plt.show()
print("Fig 1 saved")

# Figure 2: Per-class F1
fig,ax=plt.subplots(figsize=(13,5))
x=np.arange(len(RISK_LABELS)); width=0.25
for i,(m,col,label) in enumerate(zip(metrics_all,cols,approaches)):
    f1s=[m["per_class"][lbl]["f1"] for lbl in RISK_LABELS]
    bars=ax.bar(x+i*width,f1s,width,label=f"Approach {label}",color=col,alpha=0.85)
    for bar,val in zip(bars,f1s):
        if val>0.05: ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.02,
                             f'{val:.2f}',ha='center',va='bottom',fontsize=8,fontweight='bold')
ax.set_xticks(x+width); ax.set_xticklabels(RISK_LABELS,fontsize=11)
ax.set_ylabel('F1 Score'); ax.set_ylim(0,1.25)
ax.set_title('Figure 2: Per-Class F1 Score Comparison',fontweight='bold')
ax.legend(loc='upper right'); ax.grid(axis='y',alpha=0.3)
ax.axhline(0.5,color='red',linestyle=':',alpha=0.4)
plt.tight_layout(); plt.savefig('fig2_per_class_f1.png',dpi=130,bbox_inches='tight'); plt.show()
print("Fig 2 saved")

# Figure 3: Summary metrics
fig,axes=plt.subplots(1,3,figsize=(15,5))
fig.suptitle('Figure 3: Summary Metrics — Three Approaches',fontweight='bold')
for ax,mkey,mlabel in zip(axes,['accuracy','macro_f1','high_risk_fnr'],
                               ['Accuracy (↑)','Macro F1 (↑)','High-Risk FNR (↓)']):
    vals=[m[mkey] for m in metrics_all]
    bcols=['#27ae60' if v==(min(vals) if mkey=='high_risk_fnr' else max(vals))
           else '#e67e22' if v==sorted(vals,reverse=mkey!='high_risk_fnr')[1]
           else '#e74c3c' if mkey=='high_risk_fnr' else '#3498db' for v in vals]
    bars=ax.bar(approaches,vals,color=bcols,alpha=0.85)
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                f'{val:.2%}',ha='center',va='bottom',fontsize=10,fontweight='bold')
    ax.set_ylim(0,1.25); ax.set_title(mlabel,fontweight='bold')
    ax.set_xticklabels(approaches,fontsize=8,rotation=10); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('fig3_summary_metrics.png',dpi=130,bbox_inches='tight'); plt.show()
print("Fig 3 saved")

# Figure 4: Confidence
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Figure 4: Confidence per Case (green=correct, red=incorrect)',fontweight='bold')
for ax,results,title,col in zip(axes,[results_a,results_b,results_c],approaches,cols):
    df=pd.DataFrame(results)
    bcols=['#27ae60' if c else '#e74c3c' for c in df['correct']]
    ax.bar(df['id'].astype(str),df['confidence'].fillna(0.5),color=bcols,alpha=0.9)
    ax.set_ylim(0,1.15); ax.set_xlabel('Case ID'); ax.set_ylabel('Confidence')
    ax.set_title(f"Approach {title}",fontweight='bold',color=col)
    ax.axhline(0.7,color='navy',linestyle='--',alpha=0.4,linewidth=1.2)
    for bar,gt_l in zip(ax.patches,df['ground_truth']):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.02,gt_l[:3],
                ha='center',va='bottom',fontsize=7)
    ax.legend(handles=[mpatches.Patch(color='#27ae60',label='Correct'),
                       mpatches.Patch(color='#e74c3c',label='Incorrect')],
              loc='lower right',fontsize=7); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('fig4_confidence.png',dpi=130,bbox_inches='tight'); plt.show()
print("Fig 4 saved")

# Figure 5: Radar
cats=['Accuracy','Macro F1','Weighted F1','Macro P','Macro R','Hi-Risk
Sensitivity']
N=len(cats); angles=[n/float(N)*2*3.14159 for n in range(N)]+[0]
fig,ax=plt.subplots(figsize=(8,8),subplot_kw=dict(polar=True))
for m,col,label in zip(metrics_all,cols,approaches):
    vals=[m['accuracy'],m['macro_f1'],m['weighted_f1'],m['macro_p'],m['macro_r'],m['high_risk_sens']]
    vals+=vals[:1]
    ax.plot(angles,vals,'o-',linewidth=2,color=col,label=f"Approach {label}",alpha=0.9)
    ax.fill(angles,vals,alpha=0.12,color=col)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats,size=10,fontweight='bold')
ax.set_ylim(0,1); ax.set_title('Figure 5: Multi-Dimensional Radar',fontweight='bold',pad=20)
ax.legend(loc='upper right',bbox_to_anchor=(1.35,1.15),fontsize=9)
plt.tight_layout(); plt.savefig('fig5_radar.png',dpi=130,bbox_inches='tight'); plt.show()
print("Fig 5 saved")

# Figure 6: Heatmap
import matplotlib as mpl
case_labels=[f"C{c['id']}\n({c['ground_truth'][:3]})" for c in CASES]
correct_matrix=np.array([[1 if p==g else 0 for p,g in zip(pa,gt)] for pa in preds_all])
fig,ax=plt.subplots(figsize=(14,4))
cmap=mpl.colors.ListedColormap(['#e74c3c','#27ae60'])
sns.heatmap(correct_matrix,annot=True,fmt='d',cmap=cmap,xticklabels=case_labels,
            yticklabels=approaches,ax=ax,cbar=False,linewidths=0.5,linecolor='white',
            annot_kws={"size":13,"weight":"bold","color":"white"})
ax.set_title('Figure 6: Accuracy Heatmap (1=Correct, 0=Incorrect)',fontweight='bold')
ax.set_xlabel('Case ID (Ground Truth Label)')
ax.legend(handles=[mpatches.Patch(color='#27ae60',label='Correct'),
                   mpatches.Patch(color='#e74c3c',label='Incorrect')],
          loc='upper right',bbox_to_anchor=(1.13,1.0))
plt.tight_layout(); plt.savefig('fig6_accuracy_heatmap.png',dpi=130,bbox_inches='tight'); plt.show()
print("Fig 6 saved")
print("\nAll 6 figures generated!")

## Save Outputs

In [ ]:
# Save JSON
with open('benchmark_results.json','w') as f:
    json.dump({"approach_a":{"metrics":metrics_a,"results":results_a},
               "approach_b":{"metrics":metrics_b,"results":results_b},
               "approach_c":{"metrics":metrics_c,"results":results_c}},f,indent=2,default=str)
print("Saved: benchmark_results.json")

# Save CSV
rows=[]
for case,pa,pb,pc in zip(CASES,preds_a,preds_b,preds_c):
    g=case['ground_truth']
    rows.append({"Case_ID":case["id"],"Ground_Truth":g,
                 "A_Zero_Shot":pa,"A_Correct":pa==g,
                 "B_FewShot_CoT":pb,"B_Correct":pb==g,
                 "C_MentalBERT":pc,"C_Correct":pc==g})
pd.DataFrame(rows).to_csv('per_case_comparison.csv',index=False)
print("Saved: per_case_comparison.csv")
print("\nDone! All outputs saved.")